In [1]:
import pandas as pd
import numpy as np
import matplotlib
import ast
#matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

#폰트 설정(없으면 영문으로 나옴)
try:
    font_path = [f for f in fm.findSystemFonts() if 'NanumGothic' in f or 'Malgun' in f or 'AppleGothic' in f]
    if font_path:
        plt.rcParams['font.family'] = fm.FontProperties(fname=font_path[0]).get_name()
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

In [3]:
df = pd.read_csv('./funnel_instance_analysis.csv')


## 5. Funnel Instance 기반 세그먼트 분석

분석 목적  
1. **어떤 고객 집단에게**: 성별 × 연령 × 소득별로 viewed / completed 성과 비교  
2. **어떤 간격으로**: `first_viewed_instance`와 각 gap 시간의 분포 및 세그먼트 차이 확인

> 주의: `completed_rate`는 **reward 대상 오퍼(bogo, discount)** 기준으로 해석하는 것이 적절합니다.  
> `informational`은 완료 개념이 없으므로, 전환 분석에서는 제외합니다.


In [ ]:

# =========================
# 0) 분석용 데이터 복사본 준비
# =========================
analysis_df = df.copy()

age_order = ['20대 미만', '20대', '30대', '40대', '50대', '60대 이상', '누락']
income_order = ['5만 미만', '5-7.5만', '7.5-10만', '10만 이상', '누락']
gender_order = ['F', 'M', 'O', 'Unknown']

for col, order in [('age_group', age_order), ('income_group', income_order), ('gender', gender_order)]:
    analysis_df[col] = pd.Categorical(analysis_df[col], categories=order, ordered=True)

reward_df = analysis_df[analysis_df['offer_type'].isin(['bogo', 'discount'])].copy()

print('전체 instance 수 :', len(analysis_df))
print('전체 고객 수     :', analysis_df['customer_id'].nunique())
print('전체 viewed rate :', round(analysis_df['is_viewed'].mean()*100, 2), '%')
print('전체 completed rate :', round(analysis_df['is_completed'].mean()*100, 2), '%')
print('-' * 50)
print('보상형 offer instance 수 :', len(reward_df))
print('보상형 viewed rate      :', round(reward_df['is_viewed'].mean()*100, 2), '%')
print('보상형 completed rate   :', round(reward_df['is_completed'].mean()*100, 2), '%')
print('view 후 completion rate :', round(reward_df.loc[reward_df['is_viewed'], 'is_completed'].mean()*100, 2), '%')


In [ ]:

# ======================================
# 1) 성별 × 연령 × 소득 세그먼트 성과 테이블
# ======================================
segment_3d = (
    reward_df
    .groupby(['gender', 'age_group', 'income_group'], observed=False)
    .agg(
        instances=('instance_id', 'count'),
        customers=('customer_id', 'nunique'),
        viewed_rate=('is_viewed', 'mean'),
        completed_rate=('is_completed', 'mean'),
        avg_gap_to_view=('gap_to_view', 'mean'),
        med_gap_to_view=('gap_to_view', 'median'),
        avg_gap_view_to_complete=('gap_view_to_complete', 'mean'),
        med_gap_view_to_complete=('gap_view_to_complete', 'median'),
    )
    .reset_index()
)

complete_given_view = (
    reward_df[reward_df['is_viewed']]
    .groupby(['gender', 'age_group', 'income_group'], observed=False)['is_completed']
    .mean()
    .reset_index(name='complete_given_view')
)

segment_3d = segment_3d.merge(
    complete_given_view,
    on=['gender', 'age_group', 'income_group'],
    how='left'
)

for col in ['viewed_rate', 'completed_rate', 'complete_given_view']:
    segment_3d[col] = segment_3d[col] * 100

segment_3d = segment_3d.sort_values(
    ['completed_rate', 'viewed_rate', 'instances'],
    ascending=[False, False, False]
)

segment_3d.head(20)


In [ ]:

# ======================================
# 2) 표본 수가 충분한 세그먼트만 비교
#    (instances 200 이상)
# ======================================
segment_3d_valid = segment_3d[segment_3d['instances'] >= 200].copy()

top_completed = (
    segment_3d_valid[
        ['gender', 'age_group', 'income_group', 'instances', 'customers',
         'viewed_rate', 'completed_rate', 'complete_given_view',
         'med_gap_to_view', 'med_gap_view_to_complete']
    ]
    .sort_values('completed_rate', ascending=False)
    .head(10)
)

low_completed = (
    segment_3d_valid[
        ['gender', 'age_group', 'income_group', 'instances', 'customers',
         'viewed_rate', 'completed_rate', 'complete_given_view',
         'med_gap_to_view', 'med_gap_view_to_complete']
    ]
    .sort_values('completed_rate', ascending=True)
    .head(10)
)

print('[완료율 상위 세그먼트]')
display(top_completed)

print('[완료율 하위 세그먼트]')
display(low_completed)


In [ ]:

# ======================================
# 3) 상/하위 세그먼트 바 차트
# ======================================
plot_top = top_completed.copy()
plot_top['segment'] = (
    plot_top['gender'].astype(str) + ' | ' +
    plot_top['age_group'].astype(str) + ' | ' +
    plot_top['income_group'].astype(str)
)

plot_low = low_completed.copy()
plot_low['segment'] = (
    plot_low['gender'].astype(str) + ' | ' +
    plot_low['age_group'].astype(str) + ' | ' +
    plot_low['income_group'].astype(str)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(plot_top['segment'][::-1], plot_top['completed_rate'][::-1])
axes[0].set_title('완료율 상위 10개 세그먼트 (reward offers)')
axes[0].set_xlabel('Completed Rate (%)')

axes[1].barh(plot_low['segment'][::-1], plot_low['completed_rate'][::-1])
axes[1].set_title('완료율 하위 10개 세그먼트 (reward offers)')
axes[1].set_xlabel('Completed Rate (%)')

plt.tight_layout()
plt.show()


In [ ]:

# ======================================
# 4) 성별별 heatmap 함수
# ======================================
def draw_heatmap(data, value_col, gender_value, title, fmt='{:.1f}'):
    sub = data[data['gender'] == gender_value].copy()
    pv = sub.pivot(index='age_group', columns='income_group', values=value_col)
    pv = pv.reindex(index=age_order, columns=income_order)

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pv.values, aspect='auto')

    ax.set_xticks(range(len(pv.columns)))
    ax.set_xticklabels(pv.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels(pv.index)
    ax.set_title(title)

    for i in range(len(pv.index)):
        for j in range(len(pv.columns)):
            val = pv.iloc[i, j]
            text = '' if pd.isna(val) else fmt.format(val)
            ax.text(j, i, text, ha='center', va='center', fontsize=9)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(value_col)
    plt.tight_layout()
    plt.show()

# viewed rate heatmap
draw_heatmap(segment_3d_valid, 'viewed_rate', 'F', '여성 세그먼트 Viewed Rate Heatmap')
draw_heatmap(segment_3d_valid, 'viewed_rate', 'M', '남성 세그먼트 Viewed Rate Heatmap')

# completed rate heatmap
draw_heatmap(segment_3d_valid, 'completed_rate', 'F', '여성 세그먼트 Completed Rate Heatmap')
draw_heatmap(segment_3d_valid, 'completed_rate', 'M', '남성 세그먼트 Completed Rate Heatmap')



### 세그먼트 해석 포인트
- `viewed_rate`가 높은데 `completed_rate`가 낮으면 **오퍼를 보긴 보지만 행동 전환이 약한 집단**
- `viewed_rate`와 `completed_rate`가 모두 높으면 **타겟팅 우선순위가 높은 집단**
- `med_gap_to_view`, `med_gap_view_to_complete`가 짧으면 **반응 속도가 빠른 집단**


In [ ]:

# ======================================
# 5) first_viewed_instance 생성
#    - 고객별로 가장 먼저 viewed 된 instance
# ======================================
customer_seq_df = analysis_df.sort_values(['customer_id', 't_received', 'instance_id']).copy()
customer_seq_df['instance_seq'] = customer_seq_df.groupby('customer_id').cumcount() + 1

first_viewed_instance = (
    customer_seq_df[customer_seq_df['is_viewed']]
    .sort_values(['customer_id', 't_viewed', 'instance_id'])
    .groupby('customer_id', as_index=False)
    .first()
)

print('first_viewed_instance 고객 수 :', len(first_viewed_instance))
display(
    first_viewed_instance[
        ['customer_id', 'instance_id', 'instance_seq', 'offer_type', 't_received', 't_viewed', 'gap_to_view',
         'gender', 'age_group', 'income_group']
    ].head()
)


In [ ]:

# ======================================
# 6) first_viewed_instance 분포 확인
# ======================================
first_seq_dist = (
    first_viewed_instance['instance_seq']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .reset_index()
)
first_seq_dist.columns = ['instance_seq', 'pct']

first_offer_dist = (
    first_viewed_instance['offer_type']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)
first_offer_dist.columns = ['offer_type', 'pct']

display(first_seq_dist)
display(first_offer_dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(first_seq_dist['instance_seq'].astype(str), first_seq_dist['pct'])
axes[0].set_title('고객의 첫 viewed가 발생한 instance 순번')
axes[0].set_xlabel('instance_seq')
axes[0].set_ylabel('비중 (%)')

axes[1].bar(first_offer_dist['offer_type'], first_offer_dist['pct'])
axes[1].set_title('first_viewed_instance의 offer_type 분포')
axes[1].set_xlabel('offer_type')
axes[1].set_ylabel('비중 (%)')

plt.tight_layout()
plt.show()


In [ ]:

# ======================================
# 7) gap 시간 분포
# ======================================
gap_cols = ['gap_to_view', 'gap_to_complete', 'gap_view_to_complete']

for col in gap_cols:
    valid = analysis_df[col].dropna()
    print(f'[{col}]')
    print(valid.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
    print('-' * 70)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, gap_cols):
    valid = analysis_df[col].dropna()
    ax.hist(valid, bins=40)
    ax.set_title(col)
    ax.set_xlabel('hours')
    ax.set_ylabel('count')

plt.tight_layout()
plt.show()


In [ ]:

# ======================================
# 8) first_viewed_instance의 gap_to_view 분포
# ======================================
fv_gap = first_viewed_instance['gap_to_view'].dropna()

print(fv_gap.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

plt.figure(figsize=(8, 5))
plt.hist(fv_gap, bins=30)
plt.title('first_viewed_instance gap_to_view 분포')
plt.xlabel('hours from receive to first view')
plt.ylabel('count')
plt.show()


In [ ]:

# ======================================
# 9) 세그먼트별 반응 속도 비교
#    - 표본 200 이상 세그먼트만
# ======================================
speed_compare = (
    segment_3d_valid[
        ['gender', 'age_group', 'income_group', 'instances',
         'completed_rate', 'med_gap_to_view', 'med_gap_view_to_complete']
    ]
    .sort_values(['med_gap_to_view', 'med_gap_view_to_complete'])
)

display(speed_compare.head(15))

speed_compare_plot = speed_compare.copy()
speed_compare_plot['segment'] = (
    speed_compare_plot['gender'].astype(str) + ' | ' +
    speed_compare_plot['age_group'].astype(str) + ' | ' +
    speed_compare_plot['income_group'].astype(str)
)

top_fast = speed_compare_plot.head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top_fast['segment'][::-1], top_fast['med_gap_to_view'][::-1])
axes[0].set_title('반응이 빠른 세그먼트 Top 10 (median gap_to_view)')
axes[0].set_xlabel('hours')

axes[1].barh(top_fast['segment'][::-1], top_fast['completed_rate'][::-1])
axes[1].set_title('같은 세그먼트의 completed rate')
axes[1].set_xlabel('Completed Rate (%)')

plt.tight_layout()
plt.show()



## 현재 데이터 기준 핵심 인사이트

### 1) 어떤 고객 집단에게?
- **보상형 오퍼 기준 완료율 상위권은 50~60대, 중상/고소득 세그먼트**에 집중됩니다.
- 특히 **남성 60대 이상·10만 이상**, **여성 50대·10만 이상**, **여성 60대 이상·7.5~10만** 세그먼트가 매우 강하게 나타났습니다.
- 반대로 **남성 저소득층(5만 미만)**은 20대~60대 전반에서 완료율이 낮고, `gap_view_to_complete`도 더 길게 나타납니다.
- `Unknown/누락` 세그먼트는 viewed rate는 높지만 완료율이 매우 낮아, **타겟팅보다는 데이터 품질 이슈**로 먼저 점검하는 것이 좋습니다.

### 2) 어떤 간격으로?
- 고객의 **첫 viewed는 대부분 매우 이른 시점**에 발생합니다.
- 첫 viewed는 **1번째 instance에서 약 77.5%**, **2번째까지 누적 약 93.5%** 발생했습니다.
- 즉, 고객이 반응할지 여부는 **초기 1~2개의 노출에서 거의 결정**되는 경향이 있습니다.
- `first_viewed_instance`의 `gap_to_view`는 **중앙값 18시간**, **75%가 36시간 이내**, **95%가 96시간 이내**였습니다.
- 따라서 실무적으로는 **발송 후 24~48시간**을 핵심 관찰 구간으로 보고, 리마인드/채널 전략을 설계하는 해석이 가능합니다.

### 3) 오퍼 유형 관점
- 고객의 첫 viewed를 만든 오퍼 유형은 **bogo 비중이 가장 높고(약 44.9%)**, 그 다음이 **discount(약 37.8%)**였습니다.
- 즉, 첫 반응 유도 측면에서는 **bogo가 상대적으로 강한 시작점**일 가능성이 있습니다.

### 4) 다음 해석 단계 추천
- 세그먼트별로 `offer_type`까지 교차해서 보면,
  **어떤 고객에게 어떤 오퍼가 맞는지**까지 더 명확해집니다.
- 다음 단계는 아래 순서를 추천합니다.
  1. **세그먼트 × offer_type 완료율**
  2. **세그먼트 × channel_count / 채널 조합**
  3. **24h / 48h / 72h 기준 반응률 누적 비교**
